# 02 — Data Cleaning, Schema Harmonization & Parquet Pipeline (2019 vs. 2023)

This notebook runs the out-of-core **DuckDB + Polars** cleaning and schema harmonization pipeline in `src/metro_od/clean.py` to resolve every anomaly found in `01_describe_tables.ipynb`:
1. **Delimiter & Trailing Column Fixes**: Normalizes `;` (2019 & 2023 `etapas`), `,` (2023 `viajes`, dropping `Unnamed: 100`), and mixed `|;` in 2019 boarding/alighting matrices.
2. **Coordinate System Disentangling (`etapas`)**: Separates UTM Zone 19S meters (`utm_x_*`, `utm_y_*`) from WGS84 degrees (`lat_*`, `lon_*`) mixed into 2023 Metro rows.
3. **Distance & Travel Time Outlier Masking**: Masks negative sentinel route distances (`-20,601 m`) and `~6.31M m` Euclidean glitches caused by `(0, 0)` UTM fallbacks, and converts 2023 `tviaje2` (seconds) to `tviaje_min` (minutes).
4. **2019 vs. 2023 Categorical & Commune Harmonization**: Expands 2019 8-char truncated communes (`LA FLORI` -> `LA FLORIDA`) and decodes all 34 integer commune codes (`COMUNA_CODES_2023`), 12 period codes (`PERIODO_CODES_2023`), and transport modes in `viajes_2023`.
5. **2023 `.xlsb` Matrices Cleaning**: Loads `2023.04 Matriz_sub_SS_MH.xlsb` and `2023.04 Matriz_baj_SS_MH.xlsb` via `fastexcel` + `polars` with `header_row=1` and strongly-typed half-hourly columns.

In [1]:
import sys
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from metro_od.clean import (
    build_clean_etapas_query,
    build_clean_viajes_query,
    load_clean_xlsb_matrix_2023,
    export_cleaned_parquet,
)

con = duckdb.connect()
cols = ['column_name', 'column_type', 'min', 'max', 'approx_unique', 'null_percentage']

for yr in ('2019', '2023'):
    df_et = con.execute('SUMMARIZE (' + build_clean_etapas_query(yr, limit=50_000) + ')').df()
    print(f'=== Clean Etapas {yr} (50k sample) ===')
    print(df_et[cols].to_string(index=False))
    print()

=== Clean Etapas 2019 (50k sample) ===
                column_name column_type                 min                 max  approx_unique  null_percentage
                       anio     VARCHAR                2019                2019              1             0.00
                         op     INTEGER                   1                  17              8             0.00
                     netapa     INTEGER                   1                   4              4             0.00
                   tipo_dia     VARCHAR             LABORAL             LABORAL              1             0.00
            tipo_transporte     VARCHAR                 BUS           ZONA PAGA              4             0.00
    f_exp_servicioperiodots      DOUBLE                 0.0                18.0           5440             0.00
               tiene_bajada     INTEGER                   0                   1              2             0.00
              tiempo_subida   TIMESTAMP 2019-04-08 00:03:48 2019-

## 1. Harmonized `viajes` (2019 vs. 2023) — Unified Schema Describe

Both `viajes_2019` (107 raw columns) and `viajes_2023` (101 raw columns) now project into the exact same 32-column canonical OD schema with decoded commune names, time periods, transport modes, and travel times in minutes.

In [2]:
for yr in ('2019', '2023'):
    df_vj = con.execute('SUMMARIZE (' + build_clean_viajes_query(yr, limit=50_000) + ')').df()
    print(f'=== Clean & Harmonized Viajes {yr} (50k sample) ===')
    print(df_vj[cols].to_string(index=False))
    print()

=== Clean & Harmonized Viajes 2019 (50k sample) ===
            column_name column_type                 min                 max  approx_unique  null_percentage
                   anio     VARCHAR                2019                2019              1             0.00
             id_tarjeta     VARCHAR                 NaN                 NaN              0           100.00
               id_viaje     INTEGER                 NaN                 NaN              0           100.00
               tipo_dia     VARCHAR             LABORAL             LABORAL              1             0.00
       factor_expansion      DOUBLE                 0.0              3.0358           2700             0.00
               n_etapas     INTEGER                   1                   5              5             0.00
    n_etapas_sin_bajada     INTEGER                   0                   4              5             0.00
ultima_etapa_con_bajada     INTEGER                   0                   1         

## 2. Cross-Year Demand Characterization (2019 vs. 2023 Sample Comparison)

Because both years share the same schema, we can directly query `UNION ALL` across 2019 and 2023 to compare modal split, average travel time (`tviaje_min`), route distance (`distancia_ruta_m`), and stages per trip (`n_etapas`).

In [3]:
q_od_cmp = (
    'WITH v19 AS (' + build_clean_viajes_query('2019', limit=100_000) + '), '
    'v23 AS (' + build_clean_viajes_query('2023', limit=100_000) + '), '
    'combined AS (SELECT * FROM v19 UNION ALL SELECT * FROM v23) '
    'SELECT anio, tipo_transporte_1 AS modo_inicial, COUNT(*) AS viajes_muestra, '
    'ROUND(SUM(factor_expansion), 1) AS viajes_expandidos, '
    'ROUND(AVG(tviaje_min), 2) AS tviaje_promedio_min, '
    'ROUND(AVG(distancia_ruta_m) / 1000.0, 2) AS dist_ruta_promedio_km, '
    'ROUND(AVG(n_etapas), 2) AS etapas_promedio '
    'FROM combined WHERE tipo_transporte_1 IS NOT NULL '
    'GROUP BY anio, tipo_transporte_1 ORDER BY anio, viajes_expandidos DESC'
)
print(con.execute(q_od_cmp).df().to_string(index=False))

anio modo_inicial  viajes_muestra  viajes_expandidos  tviaje_promedio_min  dist_ruta_promedio_km  etapas_promedio
2019        METRO           53023            56226.4                24.95                   9.76             1.23
2019          BUS           38848            40962.5                32.28                   9.85             1.55
2019    ZONA PAGA            6861             7099.4                32.16                   7.09             1.26
2019    METROTREN            1268             1457.8                28.48                   2.93             1.54
2023        METRO           56058            57431.9                19.58                   7.42             1.18
2023          BUS           38271            37479.8                32.65                   6.20             1.52
2023    ZONA PAGA            4426             4397.3                30.97                   4.68             1.27
2023    METROTREN            1245             1283.8                18.87               

## 3. Cleaned 2023 `.xlsb` Half-Hourly Boarding & Alighting Matrices

Inspecting the top Metro stations by average workday boardings (`subidas_prom_laboral`) and alightings (`bajadas_prom_laboral`) during April 17–21, 2023.

In [4]:
df_sub23 = load_clean_xlsb_matrix_2023('matriz_subidas_2023_xlsb')
df_baj23 = load_clean_xlsb_matrix_2023('matriz_bajadas_2023_xlsb')

print(f'2023 Subidas XLSB shape: {df_sub23.shape}')
print(
    df_sub23.filter(df_sub23['paradero'].is_null())
    .select(['Comuna', 'parada SIMT', 'servicio usuario', 'TOTAL', '8:00:00', '18:00:00'])
    .sort('TOTAL', descending=True)
    .head(15)
)

print(f'\n2023 Bajadas XLSB shape: {df_baj23.shape}')
print(
    df_baj23.filter(df_baj23['parada SIMT'].is_null())
    .select(['Comuna', 'paradero', 'servicio usuario', 'TOTAL', '8:00:00', '18:00:00'])
    .sort('TOTAL', descending=True)
    .head(15)
)

2023 Subidas XLSB shape: (65188, 57)
shape: (15, 6)
┌─────────────┬──────────────────────┬──────────────────┬──────────┬─────────┬──────────┐
│ Comuna      ┆ parada SIMT          ┆ servicio usuario ┆ TOTAL    ┆ 8:00:00 ┆ 18:00:00 │
│ ---         ┆ ---                  ┆ ---              ┆ ---      ┆ ---     ┆ ---      │
│ str         ┆ str                  ┆ str              ┆ f64      ┆ f64     ┆ f64      │
╞═════════════╪══════════════════════╪══════════════════╪══════════╪═════════╪══════════╡
│ PROVIDENCIA ┆ LOS LEONES           ┆ null             ┆ 100941.2 ┆ 1847.2  ┆ 8756.0   │
│ LA CISTERNA ┆ LA CISTERNA          ┆ null             ┆ 95767.6  ┆ 7664.4  ┆ 2732.0   │
│ PROVIDENCIA ┆ TOBALABA             ┆ null             ┆ 89496.0  ┆ 1724.8  ┆ 7354.4   │
│ SANTIAGO    ┆ UNIVERSIDAD DE CHILE ┆ null             ┆ 86756.0  ┆ 1468.0  ┆ 6366.8   │
│ SANTIAGO    ┆ LOS HEROES           ┆ null             ┆ 76570.8  ┆ 1910.8  ┆ 4378.8   │
│ …           ┆ …                    ┆ …        

## 4. Export Cleaned Datasets to Compressed Parquet (`data/processed/`)

Use `export_cleaned_parquet(sample_limit=None)` to convert the entire ~20 GB raw data into fast, ZSTD-compressed Parquet files (or pass `sample_limit=100_000` for a quick test export).

In [5]:
# Set sample_limit=None to export full datasets, or sample_limit=100_000 for a quick subset
exported = export_cleaned_parquet(sample_limit=100_000)
for item in exported:
    print(f"{item['file']:35s} | {item['rows']:9d} rows | {item['size_mb']:6.2f} MB")

etapas_2019_clean.parquet           |    100000 rows |   3.79 MB
etapas_2023_clean.parquet           |    100000 rows |   3.69 MB
viajes_2019_clean.parquet           |    100000 rows |   2.72 MB
viajes_2023_clean.parquet           |    100000 rows |   2.79 MB
subidas_parada_2019_clean.parquet   |    355333 rows |   0.52 MB
bajadas_zona_2019_clean.parquet     |      3691 rows |   0.03 MB
subidas_parada_2023_clean.parquet   |     65188 rows |   2.04 MB
bajadas_parada_2023_clean.parquet   |     67857 rows |   3.50 MB
